# Thử nghiệm Đối chứng với Thư viện Scikit-Surprise

Notebook này sử dụng thư viện `scikit-surprise` để chạy các thuật toán tương tự trên cùng một tập dữ liệu MovieLens 100k.
Kết quả MAE và RMSE ở đây sẽ được dùng để đối chiếu trực tiếp với kết quả chạy từ thư mục `src/` (tự code từ đầu).

Các thuật toán cần kiểm chứng:
1. User-Based CF (Basic - Cosine)
2. User-Based CF (Means - Pearson)
3. Item-Based CF (Basic - Cosine)
4. Item-Based CF (Weighted/Adjusted Cosine - trong surprise sử dụng Pearson để gần tương đương)
5. SVD (Matrix Factorization)
\n

In [7]:
import os
import pandas as pd
from surprise import Dataset, Reader, KNNBasic, KNNWithMeans, SVD, accuracy
from surprise.model_selection import train_test_split

# Đọc dữ liệu
data_path = '../data/raw/u.data'
reader = Reader(line_format='user item rating timestamp', sep='\t')
data = Dataset.load_from_file(data_path, reader=reader)

# Chia tập Train/Test (80/20)
# Lưu ý: Seed 42 giúp kết quả chia tập giống với custom code
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print("=== ĐÁNH GIÁ THƯ VIỆN SURPRISE ===")

# 1. User-Based Basic (Cosine)
print("\n1. User-Based Basic (Cosine)")
algo_u_basic = KNNBasic(k=40, sim_options={'name': 'cosine', 'user_based': True}, verbose=False)
algo_u_basic.fit(trainset)
preds_u_basic = algo_u_basic.test(testset)
accuracy.rmse(preds_u_basic)
accuracy.mae(preds_u_basic)

# 2. User-Based Means (Pearson)
print("\n2. User-Based Means (Pearson)")
algo_u_means = KNNWithMeans(k=40, sim_options={'name': 'pearson', 'user_based': True}, verbose=False)
algo_u_means.fit(trainset)
preds_u_means = algo_u_means.test(testset)
accuracy.rmse(preds_u_means)
accuracy.mae(preds_u_means)

# 3. Item-Based Basic (Cosine)
print("\n3. Item-Based Basic (Cosine)")
algo_i_basic = KNNBasic(k=40, sim_options={'name': 'cosine', 'user_based': False}, verbose=False)
algo_i_basic.fit(trainset)
preds_i_basic = algo_i_basic.test(testset)
accuracy.rmse(preds_i_basic)
accuracy.mae(preds_i_basic)

# 4. Item-Based (Pearson - Surprise không có Adjusted Cosine, đây là phương pháp gần giống nhất)
print("\n4. Item-Based (Pearson - Thay thế cho Adjusted Cosine)")
algo_i_pearson = KNNBasic(k=40, sim_options={'name': 'pearson', 'user_based': False}, verbose=False)
algo_i_pearson.fit(trainset)
preds_i_pearson = algo_i_pearson.test(testset)
accuracy.rmse(preds_i_pearson)
accuracy.mae(preds_i_pearson)

# 5. SVD
print("\n5. SVD (Funk SVD)")
algo_svd = SVD(n_factors=20, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
algo_svd.fit(trainset)
preds_svd = algo_svd.test(testset)
accuracy.rmse(preds_svd)
accuracy.mae(preds_svd)


=== ĐÁNH GIÁ THƯ VIỆN SURPRISE ===

1. User-Based Basic (Cosine)
RMSE: 1.0194
MAE:  0.8038

2. User-Based Means (Pearson)
RMSE: 0.9492
MAE:  0.7430

3. Item-Based Basic (Cosine)
RMSE: 1.0264
MAE:  0.8104

4. Item-Based (Pearson - Thay thế cho Adjusted Cosine)
RMSE: 1.0411
MAE:  0.8332

5. SVD (Funk SVD)
RMSE: 0.9350
MAE:  0.7370


np.float64(0.7369633476148726)

In [8]:
print("\n======================================================\n")
print("=== ĐỐI CHỨNG VỚI MÔ HÌNH TỰ XÂY DỰNG TỪ ĐẦU (SCRATCH) ===\n")
print("======================================================\n")
import sys
sys.path.append("../")
from src.data_loader import load_matrix
from src.evaluation import compute_mae, compute_rmse
import pickle

test_matrix = load_matrix("../data/processed/test_matrix.npy")

with open("../models/user_cf.pkl", "rb") as f:
    user_cf = pickle.load(f)
with open("../models/item_cf.pkl", "rb") as f:
    item_cf = pickle.load(f)
    
from src.recommender import MatrixFactorizationSVD, UserBasedCollaborativeFiltering, ItemBasedCollaborativeFiltering
svd_model = MatrixFactorizationSVD()
svd_model.load_model("../models/svd_weights.pkl")

# 1. User-Based Basic (Cosine)
u_basic = UserBasedCollaborativeFiltering(k_neighbors=user_cf.k_neighbors, prediction_mode='basic')
u_basic.train_matrix = user_cf.train_matrix
u_basic.similarity_matrix = user_cf.cosine_similarity_matrix
u_basic.user_means = user_cf.user_means

# 2. User-Based Means (Pearson)
u_means = UserBasedCollaborativeFiltering(k_neighbors=user_cf.k_neighbors, prediction_mode='means')
u_means.train_matrix = user_cf.train_matrix
u_means.similarity_matrix = user_cf.pearson_similarity_matrix
u_means.user_means = user_cf.user_means

# 3. Item-Based Basic (Cosine)
i_basic = ItemBasedCollaborativeFiltering(k_neighbors=item_cf.k_neighbors)
i_basic.train_matrix = item_cf.train_matrix
i_basic.similarity_matrix = item_cf.cosine_similarity_matrix

# 4. Item-Based Trọng số (Adjusted Cosine)
i_adj_cos = ItemBasedCollaborativeFiltering(k_neighbors=item_cf.k_neighbors)
i_adj_cos.train_matrix = item_cf.train_matrix
i_adj_cos.similarity_matrix = item_cf.adjusted_cosine_similarity_matrix

print("\n1. Custom User-Based Basic (Cosine)")
print(f"RMSE: {compute_rmse(test_matrix, u_basic):.4f}")
print(f"MAE:  {compute_mae(test_matrix, u_basic):.4f}")

print("\n2. Custom User-Based Means (Pearson)")
print(f"RMSE: {compute_rmse(test_matrix, u_means):.4f}")
print(f"MAE:  {compute_mae(test_matrix, u_means):.4f}")

print("\n3. Custom Item-Based Basic (Cosine)")
print(f"RMSE: {compute_rmse(test_matrix, i_basic):.4f}")
print(f"MAE:  {compute_mae(test_matrix, i_basic):.4f}")

print("\n4. Custom Item-Based Trọng số (Adjusted Cosine)")
print(f"RMSE: {compute_rmse(test_matrix, i_adj_cos):.4f}")
print(f"MAE:  {compute_mae(test_matrix, i_adj_cos):.4f}")

print("\n5. Custom SVD")
print(f"RMSE: {compute_rmse(test_matrix, svd_model):.4f}")
print(f"MAE:  {compute_mae(test_matrix, svd_model):.4f}")




=== ĐỐI CHỨNG VỚI MÔ HÌNH TỰ XÂY DỰNG TỪ ĐẦU (SCRATCH) ===



1. Custom User-Based Basic (Cosine)
RMSE: 1.0101
MAE:  0.8048

2. Custom User-Based Means (Pearson)
RMSE: 0.9437
MAE:  0.7420

3. Custom Item-Based Basic (Cosine)
RMSE: 0.9855
MAE:  0.7752

4. Custom Item-Based Trọng số (Adjusted Cosine)
RMSE: 0.9691
MAE:  0.7665

5. Custom SVD
RMSE: 0.9349
MAE:  0.7395


### Bảng So Sánh Hiệu Năng (RMSE và MAE)
Tổng hợp lại kết quả bên trên vào chung một DataFrame để dễ quan sát và so sánh.

In [9]:
import pandas as pd

data = {
    "Mô hình": ["User-Based Basic (Cosine)", "User-Based Means", "Item-Based Basic", "Item-Based (Adj Cosine / Pearson)", "SVD"],
    "Surprise RMSE": [
        accuracy.rmse(preds_u_basic, verbose=False),
        accuracy.rmse(preds_u_means, verbose=False),
        accuracy.rmse(preds_i_basic, verbose=False),
        accuracy.rmse(preds_i_pearson, verbose=False),
        accuracy.rmse(preds_svd, verbose=False)
    ],
    "Custom RMSE": [
        compute_rmse(test_matrix, u_basic),
        compute_rmse(test_matrix, u_means),
        compute_rmse(test_matrix, i_basic),
        compute_rmse(test_matrix, i_adj_cos),
        compute_rmse(test_matrix, svd_model)
    ],
    "Surprise MAE": [
        accuracy.mae(preds_u_basic, verbose=False),
        accuracy.mae(preds_u_means, verbose=False),
        accuracy.mae(preds_i_basic, verbose=False),
        accuracy.mae(preds_i_pearson, verbose=False),
        accuracy.mae(preds_svd, verbose=False)
    ],
    "Custom MAE": [
        compute_mae(test_matrix, u_basic),
        compute_mae(test_matrix, u_means),
        compute_mae(test_matrix, i_basic),
        compute_mae(test_matrix, i_adj_cos),
        compute_mae(test_matrix, svd_model)
    ]
}

comparison_df = pd.DataFrame(data)
from IPython.display import display
display(comparison_df)


,Mô hình,Surprise RMSE,Custom RMSE,Surprise MAE,Custom MAE
0,User-Based Basic (Cosine),1.019354,1.010090,0.803799,0.804776
1,User-Based Means,0.949158,0.943703,0.742978,0.742033
2,Item-Based Basic,1.026430,0.985535,0.810381,0.775185
3,Item-Based (Adj Cosine / Pearson),1.041104,0.969113,0.833152,0.766530
4,SVD,0.934958,0.934881,0.736963,0.739464


### So sánh thực tế: Top 5 Gợi ý phim cho User 1 (dùng mô hình SVD)
Để thấy rõ hơn, chúng ta thử xem danh sách top 5 phim mà mô hình tự làm và thư viện đề xuất cho người dùng có id là 1 có tương đồng không.

In [10]:
from src.data_loader import load_movie_titles
import numpy as np

movie_titles = load_movie_titles("../data/raw/u.item")
user_id = 1
user_idx = user_id - 1

print("=== [Custom SVD] Top 5 Phim Gợi Ý ===")
unviewed_items = np.where(user_cf.train_matrix[user_idx] == 0)[0]
custom_preds = svd_model.predict_batch(user_idx, unviewed_items)
top_5_idx_custom = unviewed_items[np.argsort(custom_preds)[-5:][::-1]]

for rank, idx in enumerate(top_5_idx_custom, 1):
    m_id = int(idx) + 1
    name = movie_titles.get(m_id, f"Phim {m_id}")
    pred_score = custom_preds[np.where(unviewed_items == idx)[0][0]]
    print(f"{rank}. {name} (Dự đoán: {pred_score:.2f} sao)")

print("\n=== [Surprise SVD] Top 5 Phim Gợi Ý ===")
try:
    user_inner_id = trainset.to_inner_uid(str(user_id))
    viewed_inner_items = set([j for (j, r) in trainset.ur[user_inner_id]])
    
    surprise_preds = []
    for i in trainset.all_items():
        if i not in viewed_inner_items:
            item_raw_id = trainset.to_raw_iid(i)
            pred = algo_svd.predict(uid=str(user_id), iid=item_raw_id).est
            surprise_preds.append((int(item_raw_id), pred))
            
    surprise_preds.sort(key=lambda x: x[1], reverse=True)
    top_5_surprise = surprise_preds[:5]
    
    for rank, (m_id, pred_score) in enumerate(top_5_surprise, 1):
        name = movie_titles.get(int(m_id), f"Phim {m_id}")
        print(f"{rank}. {name} (Dự đoán: {pred_score:.2f} sao)")
except ValueError:
    print(f"Không thể tìm thấy user_id {user_id} trong trainset của Surprise.")


=== [Custom SVD] Top 5 Phim Gợi Ý ===
1. Casablanca (1942) (Dự đoán: 4.68 sao)
2. Close Shave, A (1995) (Dự đoán: 4.60 sao)
3. North by Northwest (1959) (Dự đoán: 4.46 sao)
4. One Flew Over the Cuckoo's Nest (1975) (Dự đoán: 4.44 sao)
5. Chinatown (1974) (Dự đoán: 4.42 sao)

=== [Surprise SVD] Top 5 Phim Gợi Ý ===
1. Close Shave, A (1995) (Dự đoán: 4.73 sao)
2. Schindler's List (1993) (Dự đoán: 4.65 sao)
3. L.A. Confidential (1997) (Dự đoán: 4.60 sao)
4. It's a Wonderful Life (1946) (Dự đoán: 4.59 sao)
5. One Flew Over the Cuckoo's Nest (1975) (Dự đoán: 4.59 sao)
